# Object detection: YOLO & SSD — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def conv2d(x,k,pad=0,stride=1,dilation=1):
    x=np.asarray(x,float); k=np.asarray(k,float)
    if pad: x=np.pad(x,pad)
    kh,kw=k.shape; dh=(kh-1)*dilation+1; dw=(kw-1)*dilation+1; out=[]
    for i in range(0,x.shape[0]-dh+1,stride):
        row=[]
        for j in range(0,x.shape[1]-dw+1,stride): row.append(float(np.sum(x[i:i+dh:dilation,j:j+dw:dilation]*k)))
        out.append(row)
    return np.array(out)
def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); union=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/union if union else 0
def softmax(z):
    z=np.asarray(z,float); e=np.exp(z-z.max()); return e/e.sum()
def show(a,title,cmap='viridis'):
    plt.figure(figsize=(4,3)); plt.imshow(a,cmap=cmap); plt.colorbar(); plt.title(title)
print('setup ok')

## Object detection: YOLO & SSD
Tiny CPU-only synthetic arrays for Object detection: YOLO & SSD: no downloads, no pretrained models, and every cell ends with an assert.

In [ ]:
# 1) IoU measures box overlap
a=[0,0,3,3]; b=[1,1,4,4]; v=iou(a,b); plt.figure(figsize=(3,3)); ax=plt.gca(); ax.add_patch(plt.Rectangle((0,0),3,3,fill=False)); ax.add_patch(plt.Rectangle((1,1),3,3,fill=False,color='r')); ax.set_xlim(-.5,4.5); ax.set_ylim(-.5,4.5); ax.set_aspect('equal'); plt.title(f'IoU {v:.3f}')
assert abs(v-4/14)<1e-12

In [ ]:
# 2) anchors choose the best match by IoU
anchors=np.array([[0,0,2,2],[0,0,3,3],[1,1,4,4]],float); gt=[1,1,3,3]; vals=[iou(a,gt) for a in anchors]; plt.figure(figsize=(4,3)); plt.bar(range(3),vals); plt.title('anchor IoUs')
assert int(np.argmax(vals))==1

In [ ]:
# 3) NMS removes duplicate boxes
boxes=[[0,0,3,3],[.5,.5,3.5,3.5],[5,5,7,7]]; scores=[.9,.8,.7]; keep=[]
for idx in np.argsort(scores)[::-1]:
    if all(iou(boxes[idx],boxes[j])<0.3 for j in keep): keep.append(idx)
plt.figure(figsize=(4,3)); plt.bar(range(3),scores,color=['g' if i in keep else 'r' for i in range(3)]); plt.title('NMS')
assert keep==[0,2]

In [ ]:
# 4) AP integrates precision over recall
prec=np.array([1.0,.75,.6]); rec=np.array([.33,.67,1.0]); ap=np.sum(prec*np.diff(np.r_[0,rec])); plt.figure(figsize=(4,3)); plt.step(rec,prec,where='post'); plt.title(f'AP {ap:.3f}')
assert abs(ap-0.7815)<.01

In [ ]:
# 5) matching cost selects one prediction per object
cost=np.array([[.3,1.1],[.7,.5]]); plt.figure(figsize=(3,3)); plt.imshow(cost); plt.colorbar(); plt.title('matching costs')
assert cost[0,0]+cost[1,1] < cost[0,1]+cost[1,0]